# DocFusion - Exploratory Data Analysis

This notebook explores the unified document corpus to understand:
- Data structure and field distributions
- Forgery label balance
- Vendor and total patterns
- Feature correlations for the forgery detector

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Data

In [ ]:
from docfusion.utils import load_jsonl

train_records = load_jsonl('../dummy_data/train/train.jsonl')
test_records = load_jsonl('../dummy_data/test/test.jsonl')

print(f'Training records: {len(train_records)}')
print(f'Test records: {len(test_records)}')
print(f'\nSample training record:')
train_records[0]

In [ ]:
# Flatten into a DataFrame
rows = []
for r in train_records:
    row = {
        'id': r['id'],
        'vendor': r['fields'].get('vendor'),
        'date': r['fields'].get('date'),
        'total': r['fields'].get('total'),
        'is_forged': r['label']['is_forged'],
        'fraud_type': r['label']['fraud_type'],
    }
    rows.append(row)

df = pd.DataFrame(rows)
df['total_float'] = pd.to_numeric(df['total'], errors='coerce')
df['date_parsed'] = pd.to_datetime(df['date'], errors='coerce')
df.head()

## 2. Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Forged vs genuine
label_counts = df['is_forged'].value_counts()
axes[0].bar(['Genuine (0)', 'Forged (1)'], [label_counts.get(0, 0), label_counts.get(1, 0)],
            color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Forged vs Genuine')
axes[0].set_ylabel('Count')

# Fraud types
fraud_counts = df['fraud_type'].value_counts()
axes[1].barh(fraud_counts.index, fraud_counts.values, color='#3498db')
axes[1].set_title('Fraud Type Distribution')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

## 3. Vendor Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Vendor frequency
vendor_counts = df['vendor'].value_counts()
axes[0].barh(vendor_counts.index, vendor_counts.values, color='#9b59b6')
axes[0].set_title('Vendor Frequency')

# Forgery rate by vendor
vendor_forge_rate = df.groupby('vendor')['is_forged'].mean().sort_values()
axes[1].barh(vendor_forge_rate.index, vendor_forge_rate.values, color='#e67e22')
axes[1].set_title('Forgery Rate by Vendor')
axes[1].set_xlabel('Rate')

plt.tight_layout()
plt.show()

## 4. Total Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
df['total_float'].hist(bins=20, ax=axes[0], color='#1abc9c', edgecolor='white')
axes[0].set_title('Total Amount Distribution')
axes[0].set_xlabel('Total ($)')

# Forged vs genuine totals
for label, color, name in [(0, '#2ecc71', 'Genuine'), (1, '#e74c3c', 'Forged')]:
    subset = df[df['is_forged'] == label]['total_float']
    axes[1].hist(subset, bins=15, alpha=0.6, color=color, label=name, edgecolor='white')
axes[1].set_title('Total by Forgery Status')
axes[1].set_xlabel('Total ($)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Date Patterns

In [ ]:
df['month'] = df['date_parsed'].dt.month
df['day_of_week'] = df['date_parsed'].dt.dayofweek

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby('month')['is_forged'].mean().plot(kind='bar', ax=axes[0], color='#3498db')
axes[0].set_title('Forgery Rate by Month')
axes[0].set_ylabel('Rate')

df.groupby('day_of_week')['is_forged'].mean().plot(kind='bar', ax=axes[1], color='#e74c3c')
axes[1].set_title('Forgery Rate by Day of Week')
axes[1].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], rotation=0)
axes[1].set_ylabel('Rate')

plt.tight_layout()
plt.show()

## 6. Feature Extraction & Correlation

In [ ]:
from docfusion.field_features import (
    extract_field_features,
    compute_vendor_stats,
    compute_global_stats,
    build_vendor_encoding,
    FIELD_FEATURE_NAMES,
)

vendor_stats = compute_vendor_stats(train_records)
global_stats = compute_global_stats(train_records)
vendor_encoding = build_vendor_encoding(train_records)

feat_rows = []
for rec in train_records:
    feats = extract_field_features(
        rec['fields'], vendor_stats, global_stats, vendor_encoding
    )
    feats['is_forged'] = rec['label']['is_forged']
    feat_rows.append(feats)

feat_df = pd.DataFrame(feat_rows)
feat_df.head()

In [ ]:
plt.figure(figsize=(12, 10))
corr = feat_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 7. Cross-Validation Results

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

X = feat_df[FIELD_FEATURE_NAMES].values
y = feat_df['is_forged'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

clf = GradientBoostingClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
)

# Use small cv since dummy data is tiny
cv_folds = min(3, len(y))
if cv_folds >= 2:
    scores = cross_val_score(clf, X_scaled, y, cv=cv_folds, scoring='accuracy')
    print(f'Cross-validation accuracy: {scores.mean():.3f} +/- {scores.std():.3f}')
    print(f'Per-fold scores: {scores}')
else:
    print('Not enough data for cross-validation')

## Summary

Key findings from EDA:
- The dataset contains a balanced mix of genuine and forged documents
- Multiple fraud types: price_change, layout_edit, text_edit
- Total amounts and vendor patterns provide useful features for classification
- Field-level features (z-scores, null counts, date patterns) can help distinguish forgeries